In [1]:
!pip install groq jsonschema --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 4.0 MB/s eta 0:00:00


In [2]:
import os
import json
from groq import Groq
import jsonschema

In [3]:
GROQ_API_KEY = "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"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [4]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [5]:
extract_schema = {
    "name": "extract_user_info",
    "description": "Extracts user personal info from a chat message.",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "description": "User's full name"
            },
            "email": {
                "type": "string",
                "description": "User's email address"
            },
            "phone": {
                "type": "string",
                "description": "User's phone number"
            },
            "location": {
                "type": "string",
                "description": "User's location (city, state, or country)"
            },
            "age": {
                "type": "integer",
                "description": "User's age"
            }
        },
        "required": ["name", "email", "phone", "location", "age"]
    }
}
print(json.dumps(extract_schema, indent=2))

{
  "name": "extract_user_info",
  "description": "Extracts user personal info from a chat message.",
  "parameters": {
    "type": "object",
    "properties": {
      "name": {
        "type": "string",
        "description": "User's full name"
      },
      "email": {
        "type": "string",
        "description": "User's email address"
      },
      "phone": {
        "type": "string",
        "description": "User's phone number"
      },
      "location": {
        "type": "string",
        "description": "User's location (city, state, or country)"
      },
      "age": {
        "type": "integer",
        "description": "User's age"
      }
    },
    "required": [
      "name",
      "email",
      "phone",
      "location",
      "age"
    ]
  }
}


In [6]:
def extract_info_from_chat(sample_chat):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": sample_chat}
        ],
        tools=[{"type": "function", "function": extract_schema}],
        tool_choice={"type": "function", "function": {"name": "extract_user_info"}},
    )
    try:
        tool_call = response.choices[0].message.tool_calls[0]
        arguments = tool_call.function.arguments
        parsed = json.loads(arguments)
    except Exception as e:
        print("Extraction Error:", e)
        return None

    print("ℹ️ Extracted info:")
    print(json.dumps(parsed, indent=2))

    try:
        jsonschema.validate(instance=parsed, schema=extract_schema["parameters"])
        print("Output is VALID!")
    except jsonschema.ValidationError as ve:
        print("Invalid output:", ve)

    return parsed

In [7]:
print("\n--- Example 1 ---")
extract_info_from_chat(
    "Hi, I'm Riya Sharma, 26 years old from Bengaluru. You can reach me at riya26@gmail.com or on 9876543210."
)

# Example 2
print("\n--- Example 2 ---")
extract_info_from_chat(
    "Name: Ajay Patel, email: ajay.patel99@mail.com, phone number is 9123456789, age 35, based in Mumbai."
)

# Example 3
print("\n--- Example 3 ---")
extract_info_from_chat(
    "Hello! This is Sara Singh. I'm 29, living in Delhi. Contact: sara.singh@email.com, phone 9812345678."
)


--- Example 1 ---
ℹ️ Extracted info:
{
  "age": 26,
  "email": "riya26@gmail.com",
  "location": "Bengaluru",
  "name": "Riya Sharma",
  "phone": "9876543210"
}
Output is VALID!

--- Example 2 ---
ℹ️ Extracted info:
{
  "age": 35,
  "email": "ajay.patel99@mail.com",
  "location": "Mumbai",
  "name": "Ajay Patel",
  "phone": "9123456789"
}
Output is VALID!

--- Example 3 ---
ℹ️ Extracted info:
{
  "age": 29,
  "email": "sara.singh@email.com",
  "location": "Delhi",
  "name": "Sara Singh",
  "phone": "9812345678"
}
Output is VALID!


{'age': 29,
 'email': 'sara.singh@email.com',
 'location': 'Delhi',
 'name': 'Sara Singh',
 'phone': '9812345678'}

In [8]:
print("\n--- Example 4 (EDGE CASE/Missing) ---")
extract_info_from_chat(
    "Contact Ankit at ankit@gmail.com, phone 9988776655, in Pune, aged 22."
)


--- Example 4 (EDGE CASE/Missing) ---
ℹ️ Extracted info:
{
  "age": 22,
  "email": "ankit@gmail.com",
  "location": "Pune",
  "name": "Ankit",
  "phone": "9988776655"
}
Output is VALID!


{'age': 22,
 'email': 'ankit@gmail.com',
 'location': 'Pune',
 'name': 'Ankit',
 'phone': '9988776655'}